In [ ]:
import torch
import pandas as pd
from matplotlib import pyplot as plt

training_data = torch.from_numpy(pd.read_csv('data/train.csv').to_numpy())
training_test = torch.from_numpy(pd.read_csv('data/test.csv').to_numpy())

(null): No such file or directory


In [ ]:
training_data = training_data.T
m,n = training_data.shape

Y_train = torch.eye(10)[training_data[0]].T.float() # Y is now one hot encoded
X_train = (training_data[1:n] / 255).float()

print(Y_train)
print(X_train)

print(f"Y_train current shape: {Y_train.shape}")

tensor([[0., 1., 0.,  ..., 0., 0., 0.],
        [1., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.]])
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])
Y_train current shape: torch.Size([10, 42000])


In [ ]:
# Initialise parameters

def init_params():

    W1 = torch.rand(10,784) - 0.5
    b1 = torch.rand(10,1) - 0.5
    W2 = torch.rand(10,10) - 0.5
    b2 = torch.rand(10,1) - 0.5
    W3 = torch.rand(10,10) - 0.5
    b3 = torch.rand(10,1) - 0.5

    return W1, b1, W2, b2, W3, b3

In [ ]:
# Activation Functions

def ReLU(Z):
    return torch.clamp(Z, min=0)

def ReLU_deriv(Z):
    return (Z > 0).float()

def Softmax(Z):
    A = torch.exp(Z) / torch.sum(torch.exp(Z), dim=0, keepdim=True)
    return A

# def Softmax_deriv(Z):
#     return

In [ ]:
def forward_prop(W1, b1, W2, b2, W3, b3, X):

    Z1 = W1 @ X + b1
    A1 = ReLU(Z1)
    # print(f"Z1.shape: {Z1.shape}")
    # print(f"W1.shape: {W1.shape}")
    # print(f"X.shape:  {X.shape}")
    # print(f"b1.shape: {b1.shape}")
    # print(f"A1.shape: {A1.shape}")
    # print("\n")

    Z2 = W2 @ A1 + b2
    A2 = ReLU(Z2)
    # print(f"Z2.shape: {Z2.shape}")
    # print(f"W2.shape: {W2.shape}")
    # print(f"A1.shape: {A1.shape}")
    # print(f"b2.shape: {b2.shape}")
    # print(f"A2.shape: {A2.shape}")
    # print("\n")

    
    Z3 = W3 @ A2 + b3
    A3 = Softmax(Z3)
    # print(f"Z3.shape: {Z3.shape}")
    # print(f"W3.shape: {W3.shape}")
    # print(f"A2.shape:  {A2.shape}")
    # print(f"b3.shape: {b3.shape}")
    # print(f"A3.shape: {A3.shape}")
    # print("\n")

    return Z1, A1, Z2, A2, Z3, A3

In [ ]:
def back_prop(Z1, A1, W1, Z2, A2, W2, Z3, A3, W3, X, Y):
    m = X.shape[1]

    dZ3 = A3 - Y
    dW3 = (1 / m) * dZ3 @ A2.T
    db3 = (1 / m) * torch.sum(dZ3, dim=1, keepdim=True)

    dZ2 = (W3.T @ dZ3) * ReLU_deriv(Z2)
    dW2 = (1 / m) * dZ2 @ A1.T
    db2 = (1 / m) * torch.sum(dZ2, dim=1, keepdim=True)

    dZ1 = (W2.T @ dZ2) * ReLU_deriv(Z1)
    dW1 = (1 / m) * dZ1 @ X.T
    db1 = (1 / m) * torch.sum(dZ1, dim=1, keepdim=True)

    cost = -1/m * torch.sum(Y * torch.log(A3))
    print(cost)
    
    return dW3, db3, dW2, db2, dW1, db1

In [ ]:
def update_params(W1, b1, W2, b2, W3, b3, dW1, db1, dW2, db2, dW3, db3, alpha):

    W1 -= dW1 * alpha
    b1 -= db1 * alpha
    W2 -= dW2 * alpha
    b2 -= db2 * alpha
    W3 -= dW3 * alpha
    b3 -= db3 * alpha

    return W1, b1, W2, b2, W3, b3

In [ ]:
def gradient_descent(X, Y, epochs, batch_size, alpha):
    W1, b1, W2, b2, W3, b3 = init_params(); print('parameter initialisation complete!\n')
    num_samples = X.shape[1]
    print(num_samples)

    for epoch in range(epochs):
        permutation = torch.randperm(num_samples)
        X_shuffled = X[:, permutation]
        Y_shuffled = Y[:, permutation]

        for i in range(0, num_samples, batch_size):
            Z1, A1, Z2, A2, Z3, A3 = forward_prop(W1, b1, W2, b2, W3, b3, X_shuffled[:, i:i + batch_size])
            # print('forward propagation complete!\n')

            dW3, db3, dW2, db2, dW1, db1 = back_prop(Z1, A1, W1, Z2, A2, W2, Z3, A3, W3, X_shuffled[:, i:i + batch_size], Y_shuffled[:, i:i + batch_size])
            # print('backward propagation complete!\n')
            W1, b1, W2, b2, W3, b3 = update_params(W1, b1, W2, b2, W3, b3, dW1, db1, dW2, db2, dW3, db3, alpha)
            # print('parameters updated!\n')
        
        print(f"Epoch {epoch + 1}/{epochs} complete.")
    
    return W1, b1, W2, b2, W3, b3, A3

In [33]:
W1_trained, b1_trained, W2_trained, b2_trained, W3_trained, b3_trained, A3 = gradient_descent(X_train, Y_train, 10, 200, 1e-3)

parameter initialisation complete!

42000
tensor([[4.9224e-02, 8.8262e-02, 7.6399e-03,  ..., 6.8755e-02, 6.7999e-02,
         6.0757e-02],
        [4.1558e-03, 8.2364e-02, 1.1571e-03,  ..., 1.2991e-01, 1.2695e-02,
         3.7646e-02],
        [1.0492e-03, 3.1864e-03, 3.2714e-06,  ..., 8.2750e-03, 8.6551e-04,
         2.3899e-03],
        ...,
        [7.4936e-02, 6.4517e-02, 2.9719e-03,  ..., 9.8455e-02, 2.1878e-02,
         2.5960e-02],
        [7.7810e-03, 1.6163e-01, 2.3587e-03,  ..., 1.4285e-01, 2.3103e-02,
         7.7702e-02],
        [7.6641e-01, 4.7823e-01, 9.8272e-01,  ..., 2.7188e-01, 8.2275e-01,
         7.1248e-01]])
tensor([[False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        ...,
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False]])


NameError: name 'Y_shuffled' is not defined

In [ ]:
def get_accuracy(predictions, Y):
    print(predictions, Y)
    return torch.sum(predictions == Y) / Y.size

def inference(X, W1, b1, W2, b2, W3, b3):
    _, _, _, _, _, A3 = forward_prop(W1, b1, W2, b2, W3, b3, X)
    return torch.argmax(A3, 0)

def test_inference(index, W1, b1, W2, b2, W3, b3):
    current_image = X_train[:, index, None]
    prediction = inference(X_train[:, index, None], W1, b1, W2, b2, W3, b3).item()
    label = training_data[0,index].item()
    print("Prediction: ", prediction)
    print("Label: ", label)

    current_image = current_image.reshape((28,28)) * 255
    plt.gray()
    plt.imshow(current_image, interpolation='nearest')
    plt.show()

    return prediction, label

def model_accuracy(index, W1, b1, W2, b2, W3, b3, X):
    for i in range(index):
        prediction = inference(W1, b1, W2, b2, W3, b3, X)


In [ ]:
def inference(W1, b1, W2, b2, W3, b3, X_samples):
    _, _, _, _, _, A3 = forward_prop(W1, b1, W2, b2, W3, b3, X_samples)
    return torch.argmax(A3)

def test_accuracy(W1, b1, W2, b2, W3, b3)
    